# 🧠 MyBabyAI (CodeMind) - Cloud Training System

Bu notebook, CodeMind modellerini **Google Colab** veya **Kaggle** üzerinde eğitmek için tasarlanmıştır.

### Özellikler:
- Colab/Kaggle Otomatik Algılama
- 350M-MoE Optimizasyonu
- LoRA / Full-Training Desteği
- Çoklu Veriseti Havuzu (Turkish/English/Code)
- Otomatik Checkpoint Yönetimi

In [ ]:
# @title 1. 🛠️ SİSTEM VE BAĞIMLILIKLARI KUR

# ─── ⚡ PYTORCH OOM ÖNLEYİCİ (torch import'tan ÖNCE set edilmeli) ───
import os, sys

# Bellek parçalanmasını (fragmentation) önler.
# T4/P100 gibi 16GB GPU'larda OOM hatasını dramatik azaltır.
# Bu satır her şeyden önce çalışmalı!
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✅ PYTORCH_CUDA_ALLOC_CONF set edildi.')

# Ortam Algılama
ENV = "colab" if "google.colab" in sys.modules else "kaggle" if os.path.exists("/kaggle") else "local"
print(f"Detected Environment: {ENV.upper()}")

# Paket Kurulumu (bitsandbytes>=0.41 → paged_adamw_8bit desteği)
!pip install -q transformers peft 'bitsandbytes>=0.41.0' sentencepiece huggingface_hub
!pip install -q pyyaml python-dotenv rich tqdm psutil requests datasets chromadb sentence-transformers

if ENV == "colab":
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        HF_TOKEN = None
elif ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except:
        HF_TOKEN = None
else:
    HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    !huggingface-cli login --token $HF_TOKEN
    print("✅ HuggingFace Girişi Yapıldı")


In [ ]:
# @title 2. 📂 REPO VE DİZİN AYARLARI
REPO_URL = "https://github.com/halilibrahimavsar/mybabyai.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

if not os.path.exists("mybabyai"):
    !git clone -b $BRANCH $REPO_URL
    %cd mybabyai
else:
    %cd mybabyai
    !git pull

import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

os.makedirs("models/fine_tuned", exist_ok=True)
print("✅ Çalışma dizini hazır.")


In [ ]:
# @title 3. ⚙️ EĞİTİM KONFİGÜRASYONU

model_size = "350M-MoE" # @param ["125M", "350M", "350M-MoE", "650M"]
training_mode = "full" # @param ["lora", "full"]
load_from_checkpoint = False # @param {type:"boolean"}

# 💡 T4 16GB + Full Train önerilen değerler:
#   batch_size=1, gradient_accumulation=16 → effective batch=16
#   LoRA için batch_size=4, gradient_accumulation=4 yeterlidir.
batch_size = 1 # @param {type:"integer"}
gradient_accumulation = 16 # @param {type:"integer"}
learning_rate = 5e-5 # @param {type:"number"}
epochs = 3 # @param {type:"integer"}
save_steps = 200 # @param {type:"integer"}
max_seq_length = 256 # @param {type:"integer"}  ← T4 için 256 önerilir, 512+ OOM riski

output_dir = "/content/drive/MyDrive/mybabyai_checkpoints" if ENV == "colab" else "/kaggle/working/checkpoints"

if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(output_dir, exist_ok=True)

print(f"--- {model_size} ({training_mode}) Yapılandırması Tamam ---")
print(f"    Batch: {batch_size} | GradAcc: {gradient_accumulation} | EffBatch: {batch_size*gradient_accumulation}")
print(f"    MaxSeqLen: {max_seq_length} | LR: {learning_rate} | Epochs: {epochs}")


In [ ]:
# @title 3.5. ⚡ FLASH ATTENTION / SDPA KONTROLÜ
import torch

# Flash Attention Durum Özeti:
# ┌─────────────────────────────────────────────────────────────┐
# │ T4 (Turing SM75): flash_attn v2 ÇALIŞMAZ (SM>=80 gerekir) │
# │ Bunun yerine PyTorch 2.0+ SDPA kullanıyoruz (otomatik)     │
# │ SDPA: Memory-efficient attention, %10-20 VRAM azaltır       │
# └─────────────────────────────────────────────────────────────┘

torch_ver = tuple(int(x) for x in torch.__version__.split('.')[:2])
sdpa_available = torch_ver >= (2, 0) and hasattr(torch.nn.functional, 'scaled_dot_product_attention')

# GPU mimarisi kontrolü
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    gpu_name = torch.cuda.get_device_name(0)
    is_ampere_plus = cap[0] >= 8  # A100, A10, RTX3090+
    print(f'GPU: {gpu_name} (SM {cap[0]}{cap[1]})')
    if is_ampere_plus:
        print('✅ Ampere+ GPU: Flash Attention v2 MEVCUT (tam hız, %30-40 VRAM azalışı)')
        USE_FLASH_ATTN = 'flash_attention_2'
    elif sdpa_available:
        print('⚠️  T4/Turing GPU: flash_attn v2 desteklenmiyor.')
        print('✅ PyTorch SDPA (scaled_dot_product_attention) aktif: %10-20 VRAM tasarrufu')
        USE_FLASH_ATTN = 'sdpa'
    else:
        print('ℹ️  Eski GPU/PyTorch: Standart attention kullanılacak.')
        USE_FLASH_ATTN = 'eager'
else:
    print('CPU modu: attention optimizasyonu N/A')
    USE_FLASH_ATTN = 'eager'

print(f'\n→ attn_implementation = "{USE_FLASH_ATTN}"')


In [ ]:
# @title 4. 🤖 MODEL & TRAINER BAŞLATMA
import gc
import torch

# ── VRAM temizliği ──
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | {total_gb:.1f} GB toplam | {free_gb:.1f} GB boş")

from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer
from src.utils.config import Config

config = Config()
config.set("training.output_dir", output_dir)
config.set("training.per_device_train_batch_size", batch_size)
config.set("training.gradient_accumulation_steps", gradient_accumulation)
config.set("training.learning_rate", learning_rate)
config.set("training.num_train_epochs", epochs)
config.set("training.save_steps", save_steps)
config.set("training.max_length", max_seq_length)
# Full training için gradient checkpointing zorunlu:
config.set("training.gradient_checkpointing", True)
# Flash Attention / SDPA ayarı (model yüklenirken uygulanır)
config.set("model.attn_implementation", USE_FLASH_ATTN)

model_manager = ModelManager(config)

if load_from_checkpoint:
    print("📦 Mevcut checkpoint yükleniyor...")
    model_manager.load_model("codemind", allow_fresh_fallback=True)
else:
    print(f"🌱 Sıfırdan CodeMind-{model_size} oluşturuluyor...")
    model_manager.load_fresh_model(size=model_size)

trainer = LoRATrainer(model_manager, config)
# NOT: prepare_model_for_training, train_from_pool içinde otomatik çağrılır.

print(f"✅ {model_manager.model_name} Eğitime Hazır!")


In [ ]:
# @title 5. 🎯 DATASET HAVUZU (TÜRKÇE ODAKLI)

# 💡 T4 için max_samples 5000-10000 arasında tutulması önerilir.
#    Daha fazlası işleme süresi uzatır ama OOM'a neden olmaz (tokeniz. CPU'da yapılır).
dataset_pool = [
    {
        "name": "🇹🇷 Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "max_samples": 5000
    },
    {
        "name": "🇹🇷 Turkish Alpaca",
        "type": "huggingface",
        "dataset_key": "turkish_alpaca",
        "max_samples": 3000
    }
]

print(f"📊 Dataset havuzu: {len(dataset_pool)} kaynak hazır.")


In [ ]:
# @title 6. 🚀 EĞİTİMİ BAŞLAT

# Son VRAM temizliği — eğitimden hemen önce
import gc, torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f"🔋 Eğitim öncesi boş VRAM: {free_gb:.1f} GB")

try:
    trainer.train_from_pool(
        dataset_pool,
        training_type=training_mode,
        max_length=max_seq_length,
        use_notebook_callback=True,
    )
except KeyboardInterrupt:
    print("\n🛑 Eğitim kullanıcı tarafından durduruldu.")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print("\n❌ CUDA OOM! Öneri:")
        print("   1. max_seq_length'i 128'e düşür ve yeniden başlat")
        print("   2. training_mode='lora' yap")
        print("   3. Kernel'i yeniden başlatıp Cell 1'den itibaren çalıştır")
        torch.cuda.empty_cache()
        gc.collect()
    raise


In [ ]:
# @title 7. 🧪 TEST (INFERENCE)
prompt = "Merhaba, nasılsın?" # @param {type:"string"}
max_new_tokens = 100 # @param {type:"integer"}

print("Generating...")
response = model_manager.generate(prompt, max_new_tokens=max_new_tokens)
print(f"\n🤖 AI: {response}")


## 📖 Yardımcı Rehber

### Checkpoint Dosyalarını Almak:
1. **Colab:** `/content/drive/MyDrive/mybabyai_checkpoints` klasörüne (Google Drive) otomatik kaydedilir.
2. **Kaggle:** Eğitim bitince sağ üstten **"Save Version"** → **"Quick Save"**. Çıktı sekmesinden indirebilirsiniz.

### OOM Hatası Alırsanız (Kontrol Listesi):
| Adım | Aksiyon |
|------|---------|
| 1 | `max_seq_length`'i **128** yap (en etkili) |
| 2 | `gradient_accumulation`'ı **32**'ye yükselt |
| 3 | `training_mode`='**lora**' yap |
| 4 | Kernel'i **Restart** edip Cell 1'den başa al |
| 5 | Colab'da **Runtime → Disconnect and delete runtime** sonra yeniden bağlan |

### T4 (16GB) İçin Onaylı Full-Train Ayarları:
```
batch_size = 1
gradient_accumulation = 16   # effective batch = 16
max_seq_length = 256
training_mode = "full"
```
